# From Business Question to Evidence
## A step-by-step lesson for Group Project Parts 1 and 2

**Course:** MGS 3101 - Business Analytics  
**Purpose:** Learn how to load, check, and explore data before making claims or fitting models.  
**Before class:** Bring the Group Project notebook and the data guide for your team's chosen track.

This notebook contains a **self-contained, synthetic practice example**. Run its cells in order; no project CSV files are needed here. Complete and submit your team's actual work in the **Group Project notebook**, using the files and track assigned to your team. The practice example is not evidence about real customers.

### Learning outcomes
By the end, you should be able to:
1. Translate a business question into a row unit, outcomes, and a prediction-time boundary.
2. Check joins, missing values, duplicates, types, and plausible ranges before analysis.
3. Describe event rates, distributions, scatter plots, and correlations with appropriate context.
4. Write evidence-based answers that acknowledge uncertainty, potential leakage, and limits to causal interpretation.
5. Transfer the process to the actual requirements and Git checkpoints for Parts 1 and 2.

### Suggested 60-minute agenda
| Minutes | Focus |
|---:|---|
| 0-8 | Frame the decision, row, outcomes, and timing |
| 8-25 | Part 1: load, join, and check the data |
| 25-43 | Part 2: summarize and visualize patterns |
| 43-55 | Turn results into defensible written answers |
| 55-60 | Team checklist and exit ticket |

**Thinking habit:** For every result, ask: What is being counted? What comparison is being made? What might explain it? What would I need to check next?

## Step 1: Frame the question before opening a file

The project asks your team to investigate a business decision, not just produce charts. **Part 1** establishes what the data represent and whether the modeling table is trustworthy. **Part 2** describes patterns that may be useful to investigate later. Neither part requires fitting a model.

For your own track, write down:
1. **Decision:** What decision might the business make, and for whom?
2. **Row unit:** Is one row an account, renewal, contacted person, or closed opportunity? A count of rows is only a count of that unit.
3. **Two outcomes:** What is the numeric outcome and its unit? What does a 1 mean for the yes/no event? A 0?
4. **Prediction moment:** When would the decision be made? What was knowable then, rather than only after the outcome?
5. **Population:** Who is included or excluded, and what time period does the dataset cover?

### Track reference: verify the details in your data guide
| Track | One modeling row | Numeric outcome | Yes/no event (`1`) |
|---|---|---|---|
| `saas_renewal` | Contract renewal | Customer health score (points) | Churned / did not renew |
| `account_growth` | Customer account | Customer health score (points) | Bought a cross-sell product |
| `nonprofit` | Person contacted | Donation amount (USD), **donors only** | Donated |
| `sales_activity` | Closed sales opportunity | Opportunity amount (USD) | Won the deal |
| `sales_pipeline` | Closed won or lost opportunity | Days from creation to close (days) | Won the deal |

The nonprofit numeric analysis is restricted to donors; the yes/no event rate describes all contacted people in its modeling table. The sales pipeline table excludes open opportunities. These differences change what a finding can represent.

**Pause:** If your row is a renewal rather than an account, when would it be misleading to call the row count a customer count?

## Step 2: Create a practice dataset

Our example resembles the SaaS renewal track, but is deliberately separate from the assignment data. A fictional software company wants to understand which expiring contracts may need attention. One row in the intended modeling table is **one renewal**; here there is one renewal per account. The yes/no outcome is `Churned` (1 = did not renew; 0 = renewed), and the numeric outcome is `Customer_Health_Score` (points on a 0-100 scale).

The example includes a renewal table and an account table so we can practice checking a join. Customer features are intended to be known at a pre-renewal snapshot; churn is observed later. All values and relationships are made up for teaching.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rng = np.random.default_rng(3101)
pd.set_option('display.max_columns', 30)
plt.style.use('seaborn-v0_8-whitegrid')

n = 180
account_ids = [f'A{i:04d}' for i in range(1, n + 1)]
company_size = rng.choice(['Small', 'Midmarket', 'Enterprise'], n, p=[0.45, 0.35, 0.20])
industry = rng.choice(['Technology', 'Retail', 'Manufacturing'], n)
adoption = np.clip(rng.normal(60, 19, n), 0, 100).round(1)
support_cases = rng.poisson(2.5, n)
health = np.clip(25 + 0.7 * adoption - 3 * support_cases + rng.normal(0, 8, n), 0, 100).round(1)
churn_score = 1.0 - 0.035 * adoption + 0.22 * support_cases
churn_probability = 1 / (1 + np.exp(-churn_score))
churned = rng.binomial(1, churn_probability)

practice_renewals = pd.DataFrame({
    'Renewal_ID': [f'R{i:04d}' for i in range(1, n + 1)],
    'Account_ID': account_ids,
    'Product_Adoption_Percent': adoption,
    'Support_Cases_Last_90_Days': support_cases,
    'Customer_Health_Score': health,
    'Churned': churned,
})
practice_accounts = pd.DataFrame({
    'Account_ID': account_ids,
    'Company_Size': company_size,
    'Industry': industry,
})
print('Renewal rows:', len(practice_renewals), '| Account rows:', len(practice_accounts))
display(practice_renewals.head())
display(practice_accounts.head())

## Part 1: Load and check the data

### Step 3: Identify files, keys, and join expectations

In the actual project, select `TRACK` and, if applicable, `VARIANT` in the setup cell. Read the matching data guide, place the listed files where the project notebook can read them, and run its setup and `load_track` cells. The loader constructs `cfg` and the modeling table `data` for the chosen track.

Before a join, identify the key and expected relationship. In this **practice** example the renewal table is the left table and `Account_ID` is unique in both tables, so the intended relationship is one-to-one. `validate='one_to_one'` checks key uniqueness; it **does not** prove that every key matched. A left join keeps left-table rows, but duplicates on the right can multiply rows without a validation check.

The code below uses a merge indicator to check matches as well as row counts. In your actual project, a relationship may differ: for example, sales opportunities can share an account, and holdings or activity files can have many rows per key. Aggregate detail records when the project specifies that before joining them to a one-row-per-entity table.

In [ ]:
print('Duplicate renewal IDs:', practice_renewals['Renewal_ID'].duplicated().sum())
print('Duplicate account keys in renewals:', practice_renewals['Account_ID'].duplicated().sum())
print('Duplicate account keys in accounts:', practice_accounts['Account_ID'].duplicated().sum())

practice_data = practice_renewals.merge(
    practice_accounts, on='Account_ID', how='left', validate='one_to_one', indicator=True
)
print('\nRows before join:', len(practice_renewals), '| after join:', len(practice_data))
print('Join match counts:')
display(practice_data['_merge'].value_counts())
practice_data = practice_data.drop(columns='_merge')

**Interpret:** Matching row counts are reassuring, but they are not enough: a left join can preserve every left row while leaving unmatched right-side information missing. The merge indicator checks left-side matches; a left join does not reveal right-only records. Compare key sets separately if those matter. If a check fails, investigate file definitions, key formats, missing keys, and the intended relationship before choosing a correction. Do not silently drop records.

**Your turn:** What are the join keys and expected relationship(s) in your track? What evidence shows whether rows were lost, multiplied, or left unmatched?

### Step 4: Inspect structure, missingness, and plausible values

A valid join does not guarantee valid measurements. Check the table's dimensions and sample rows, column types, missing values, unique row IDs, and numeric summaries. Ask what each blank means before filling or deleting it. An unusual number is a question to investigate, not automatically a mistake.

In the assignment, the printed missing count covers the configured model columns, **not necessarily the entire table**. Read `load_track` to understand any transformations. For example, the nonprofit loader flags prior-donation history before filling missing months with zero; zero has meaning only together with that flag. It also fills missing donation amounts with zero. Other tracks have different logic; report what actually happened in yours.

In [ ]:
print('Rows and columns:', practice_data.shape)
display(practice_data.head())
print('Column types:')
display(practice_data.dtypes.to_frame('dtype'))
print('Missing values:')
display(practice_data.isna().sum().to_frame('missing_count'))
print('Duplicate renewal IDs:', practice_data['Renewal_ID'].duplicated().sum())
print('Numeric summaries:')
display(practice_data.describe().T.round(2))
print('Adoption outside 0-100:', (~practice_data['Product_Adoption_Percent'].between(0, 100)).sum())
print('Negative support-case counts:', (practice_data['Support_Cases_Last_90_Days'] < 0).sum())
print('Observed churn codes:', sorted(practice_data['Churned'].dropna().unique()))

**Think like an analyst:** Are the observed minimum and maximum plausible given the field definition? Does a missing field mean zero, unknown, or not applicable? Would removing incomplete rows change the population? Do the observed event codes support interpreting a mean as a share? Consult the guide and report both the finding and any unresolved uncertainty.

In the practice data, health is explicitly generated partly from adoption and support cases. A strong relationship among these fields is therefore expected and should not be presented as independent evidence of a causal effect. For your track, inspect field definitions and timing: outcome-derived fields, post-decision measurements, or inputs that construct a target may make a later model misleading.

### Step 5: Describe the event and the numeric outcome

A **count** tells us how many; a **rate** tells us what fraction. If the event is coded 1 and no event is coded 0, its mean is the event rate **among nonmissing labels**. Check missing labels and report both numerator and denominator. Class balance matters: for example, when churn is rare, a classifier that always predicts no churn may have high accuracy while finding no churned contracts.

In [ ]:
event_counts = practice_data['Churned'].value_counts(dropna=False).sort_index()
event_count = practice_data['Churned'].sum()
labeled_count = practice_data['Churned'].count()
print('Churn outcomes (0 = renewed, 1 = churned):')
display(event_counts.to_frame('renewals'))
print(f'Churn rate: {event_count / labeled_count:.1%} ({event_count} of {labeled_count} labeled renewals)')
print('Missing outcome labels:', practice_data['Churned'].isna().sum())
print('\nHealth-score distribution (points):')
display(practice_data['Customer_Health_Score'].describe().to_frame('health_score'))

**Pause:** Compare the mean and median health score and inspect the range. Are there unusual values worth checking? Would a percentage without its denominator make the finding harder to evaluate? What additional comparison would help a decision-maker?

## Part 2: Explore the data

### Step 6: Compare event rates across groups

The project plots the event rate for two configured categorical variables (`cfg['explore_cats']`). Calculate group counts and event counts too: a high rate based on a handful of observations is less stable than a similar rate based on hundreds. Missing category values and missing event labels can affect denominators; check them before interpreting results.

In our practice example, compare churn by company size and industry. `count` is the number of **nonmissing event labels**, `sum` is the number of churn events, and `mean` is their event rate.

In [ ]:
practice_categories = ['Company_Size', 'Industry']
for category in practice_categories:
    summary = practice_data.groupby(category, dropna=False).agg(
        labeled_renewals=('Churned', 'count'), churned=('Churned', 'sum'), churn_rate=('Churned', 'mean')
    ).sort_values('churn_rate', ascending=False)
    print(f'Churn by {category}:')
    display(summary.round(3))

size_summary = practice_data.groupby('Company_Size').agg(
    labeled_renewals=('Churned', 'count'), churned=('Churned', 'sum'), churn_rate=('Churned', 'mean')
).sort_values('churn_rate')
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(size_summary.index, size_summary['churn_rate'], color='teal')
ax.set_xlim(0, 1.15)
ax.set_xlabel('Share of labeled renewals that churned')
ax.set_title('Observed churn rate by company size (practice data)')
for bar, (_, row) in zip(bars, size_summary.iterrows()):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{row['churn_rate']:.0%} (n={int(row['labeled_renewals'])})", va='center')
plt.tight_layout()
plt.show()

**Interpretation practice:** Identify the highest and lowest observed rates, and include each group's number of events and labeled renewals. Are the groups similarly sized? Does a higher rate mean that being in a group caused churn? No: other characteristics and random variation may contribute. A difference may be worth investigating without proving a cause or justifying immediate action.

### Step 7: Examine a numeric relationship

The assignment plots the configured numeric outcome against one configured numeric feature. Our example compares adoption (0-100 score) with health score (points); colors show the binary outcome. Read the axis labels and units, then look for direction, shape, spread, overlap, and possible outliers. Ask whether a straight line seems like a reasonable approximation or whether curvature, clusters, or wide variation suggest caution.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for event, label, color in [(0, 'Renewed (0)', 'tomato'), (1, 'Churned (1)', 'teal')]:
    group = practice_data[practice_data['Churned'] == event]
    ax.scatter(group['Product_Adoption_Percent'], group['Customer_Health_Score'],
               label=label, color=color, alpha=0.55)
ax.set_xlabel('Product adoption score (0-100)')
ax.set_ylabel('Customer health score (points)')
ax.set_title('Health vs. adoption by renewal outcome (practice data)')
ax.legend()
plt.tight_layout()
plt.show()

**Questions:** Does the cloud trend upward or downward? How much spread is there at similar adoption scores? Do event and non-event points overlap? Is a straight-line approximation plausible? Remember: this health score was constructed partly from adoption, so the apparent relationship should not be treated as evidence that improving adoption would cause a health change.

In the nonprofit track, `cfg['num_data']` contains donors only, so the scatter describes donors and all points have `Donated = 1`. It cannot compare donors with non-donors. The assignment's category-rate plots instead use the full modeling table.

### Step 8: Read a correlation table carefully

Pearson correlation ranges from -1 to 1 and describes the direction and strength of a **linear association** between two numeric variables. Near zero does not rule out a curved relationship. A binary 0/1 column can be correlated with a numeric column, but correlation does not establish cause, adjust for other variables, or guarantee useful predictions. Missing values can mean different pairs use different numbers of rows.

In [ ]:
practice_numeric = ['Product_Adoption_Percent', 'Support_Cases_Last_90_Days',
                    'Customer_Health_Score', 'Churned']
practice_corr = practice_data[practice_numeric].corr()
print('Correlations in the full practice modeling table:')
display(practice_corr.round(2))
print('Correlations with churn, sorted by absolute strength:')
with_churn = practice_corr['Churned'].drop('Churned').dropna()
display(with_churn.reindex(with_churn.abs().sort_values(ascending=False).index).to_frame('correlation'))

**Interpretation:** Identify a positive and a negative association if present, with their values. Which appears strongest in absolute terms? Is it available when the decision is made, or is it an outcome or constructed from one? The project correlation table uses `data` (the full modeling table), while its scatter uses `cfg['num_data']`; in the nonprofit track these are different populations. Categorical variables do not appear in this numeric correlation matrix.

## Step 9: Turn observations into defensible answers

A useful analytical statement has four parts: **population + comparison + evidence + qualification**. For example:

> In the practice renewals, [group A] had an observed churn rate of [events / labeled renewals = rate], compared with [group B's count and rate]. The difference is descriptive; group sizes, other characteristics, and random variation should be considered before acting.

Replace the brackets with actual displayed values; do not copy this example as a finding. For a scatter plot, state the x/y variables and units, direction, spread, whether a line seems reasonable, and any important limitation. For correlations, include the pair and approximate value, not just 'strong' or 'weak.'

**Practice:** Write three distinct observations from the outputs above: (1) a group-rate comparison with counts, (2) a numeric scatter pattern, and (3) a correlation with churn. For each, add one reason to be cautious and one follow-up question. Which variables look worth investigating for the numeric and yes/no outcomes? Check availability and construction before calling them candidate predictors; exploration alone does not establish predictive performance.

## Step 10: Transfer the workflow to your Group Project

Use the **project notebook**, not this practice dataset, to produce your team's results. Choose the correct `TRACK` and permitted `VARIANT`; run its cells from the beginning. Do not redefine `load_track` here or assume the synthetic practice results apply to your track.

### Part 1: load and check
1. Start on `part1-data` as directed. Read the chosen track's guide and identify every required file and join key.
2. Run the imports, settings, loader, and numeric-summary cells. Inspect `cfg['title']`, `cfg['id_col']`, `cfg['num_target']`, `cfg['num_unit']`, `cfg['bin_target']`, `cfg['event']`, and `cfg['num_data']`.
3. Check source and joined row counts, uniqueness at the intended row grain, join matches, types, missingness, and plausible ranges. Read the loader's joins, filters, and fill/flag logic; explain them using evidence rather than assuming a join succeeded because a cell ran.
4. Answer **every Answer 1 prompt**: row unit; numeric and yes/no outcomes and meaning/units; event share and why class balance matters; join results and missing-value handling; and a timing or target-construction risk.
5. Save, commit, push `part1-data`, and open a pull request as the project specifies.

### Part 2: explore
1. Run the two category-rate plots, the numeric scatter, and the correlation table. Inspect axis labels, units, row populations, and color meanings.
2. Calculate group counts and event counts alongside rates; check whether missing labels or categories affect denominators.
3. Answer **Answer 2** with at least three evidence-backed patterns, promising variables for **each** later model, and whether a straight line looks reasonable for the numeric outcome. Distinguish an observed association from a causal or predictive conclusion.
4. Save, commit, push, request teammate review, and merge `part1-data` into `main` before the other parts begin, as directed in the project.

The optional cells below are **project-notebook snippets**, not runnable in this lesson until the project's setup has created `cfg` and `data`. They add evidence for the written answers; they do not replace reading the guide or checking the source files.

### Optional project-notebook bridge
```python
# Run after Part 1 setup in the GROUP PROJECT notebook.
print('Track:', cfg['title'])
print('Modeling rows:', len(data), '| unique row IDs:', data[cfg['id_col']].nunique())
print('Duplicate row IDs:', data[cfg['id_col']].duplicated().sum())
print('Missing by column:')
display(data.isna().sum().to_frame('missing_count'))
print('Event counts, including missing:')
display(data[cfg['bin_target']].value_counts(dropna=False))
print('Numeric outcome population:', len(cfg['num_data']))

# Part 2: counts and rates for the actual configured categories.
for col in cfg['explore_cats']:
    print('Event summary by', col)
    display(data.groupby(col, dropna=False).agg(
        labeled_rows=(cfg['bin_target'], 'count'), events=(cfg['bin_target'], 'sum'),
        event_rate=(cfg['bin_target'], 'mean')
    ).sort_values('event_rate', ascending=False))
```
These checks do **not** by themselves confirm joins: compare the original sources and join indicators or key sets appropriate to your track. A unique ID also does not prove that the correct population was retained. `count` excludes missing labels; report missing labels separately. The group summary assumes the binary event is coded 0/1, which you should verify first.

## Step 11: Team worksheet and exit ticket

### Complete with your team
- Track, variant, and guide consulted: ___
- Business decision and intended prediction moment: ___
- One modeling row and the population represented: ___
- Numeric outcome, units, and any subset restriction: ___
- Yes/no outcome, meaning of 1 and 0, event count / labeled count / rate: ___
- Source files, join keys, and expected relationships: ___
- Evidence about row loss, duplication, unmatched keys, and missing-value handling: ___
- Potential post-outcome or target-construction issue to investigate: ___
- Three Part 2 patterns, each with numerical evidence and a limitation: ___
- Variables worth investigating for each outcome and whether the numeric scatter looks linear: ___

### Exit ticket
1. Why can a left join preserve row count yet still be incomplete?
2. Why should a group rate be accompanied by its event count and denominator?
3. What can a scatter plot and a correlation each show, and what can neither prove?
4. Give one example of a field whose timing or construction could make later predictions misleading.
5. What can your team honestly conclude if the observed patterns are weak or the data do not answer the business question?

**Takeaway:** Good analysis connects a clear business question to a defensible population, checked data, evidence, and appropriately limited claims.